### RAG Pipeline- Data Ingestion to Vector DB Pipelein

In [1]:
import os
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/tmp/ipykernel_1242431/3712246158.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader, PyMuPDFLoader
/home/gauravgupta/codebase/agentic-ai-krish-naik/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### read all pdf files from a directory and load them into a list of documents
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")



Found 3 PDF files to process

Processing: BASIC_DSA_TRCIKS.pdf
  ✓ Loaded 12 pages

Processing: git.pdf
  ✓ Loaded 13 pages

Processing: Tricks_for_DSA_Problem_Solving.pdf
  ✓ Loaded 23 pages

Total documents loaded: 48


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'Aspose.PDF for .NET 21.5.0', 'creator': 'Aspose Ltd.', 'creationdate': '2023-09-22T17:37:59+05:30', 'moddate': '2023-09-22T18:03:42+05:30', 'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'BASIC_DSA_TRCIKS.pdf', 'file_type': 'pdf'}, page_content="Tips & Tricks\nfor \nSolving a Coding Question \nDuring Interviews\nLogicmojo Academy\nDon't solve randomly 500+ Leetcode problems \n                 but focus on tricks & techniques for solving problems"),
 Document(metadata={'producer': 'Aspose.PDF for .NET 21.5.0', 'creator': 'Aspose Ltd.', 'creationdate': '2023-09-22T17:37:59+05:30', 'moddate': '2023-09-22T18:03:42+05:30', 'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2', 'source_file': 'BASIC_DSA_TRCIKS.pdf', 'file_type': 'pdf'}, page_content="Technique  # 1\nIf we are dealing with\ntop/maximum/minimum/closest \n'K' elements among 'N' elem

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Total chunks created: {len(split_docs)}")
    
    # show example of the first chunk
    if split_docs:
        print(f"\nExample of first chunk:\n{split_docs[0].page_content[:500]}...")
    
    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents, chunk_size=1000, chunk_overlap=200)
chunks

Total chunks created: 35

Example of first chunk:
Tips & Tricks
for 
Solving a Coding Question 
During Interviews
Logicmojo Academy
Don't solve randomly 500+ Leetcode problems 
                 but focus on tricks & techniques for solving problems...


[Document(metadata={'producer': 'Aspose.PDF for .NET 21.5.0', 'creator': 'Aspose Ltd.', 'creationdate': '2023-09-22T17:37:59+05:30', 'moddate': '2023-09-22T18:03:42+05:30', 'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'BASIC_DSA_TRCIKS.pdf', 'file_type': 'pdf'}, page_content="Tips & Tricks\nfor \nSolving a Coding Question \nDuring Interviews\nLogicmojo Academy\nDon't solve randomly 500+ Leetcode problems \n                 but focus on tricks & techniques for solving problems"),
 Document(metadata={'producer': 'Aspose.PDF for .NET 21.5.0', 'creator': 'Aspose Ltd.', 'creationdate': '2023-09-22T17:37:59+05:30', 'moddate': '2023-09-22T18:03:42+05:30', 'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf', 'total_pages': 12, 'page': 1, 'page_label': '2', 'source_file': 'BASIC_DSA_TRCIKS.pdf', 'file_type': 'pdf'}, page_content="Technique  # 1\nIf we are dealing with\ntop/maximum/minimum/closest \n'K' elements among 'N' elem

### embedding and vector store DB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the EmbeddingManager with a specified model for generating embeddings and set up a ChromaDB client.

        Args:
            model_name : Hugging Face model name for generating embeddings. Default is "all-MiniLM-L6-v2".
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the embedding model."""
        try:
            print(f"end=Loading model '{self.model_name}'...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model '{self.model_name}' loaded successfully.")
            print(f"Model details: {self.model.get_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model '{self.model_name}': {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts : List of strings to generate embeddings for. 
        
        Returns:
            np.ndarray : Array of embeddings.
        """
        if not self.model:
            raise ValueError("Model is not loaded. Call _load_model() first.")
        
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, convert_to_numpy=True)
            print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
            return embeddings
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise
        
## initialize the embedding manager and generate embeddings for the chunks
embedding_manager = EmbeddingManager()
embedding_manager

end=Loading model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 829.61it/s]


Model 'all-MiniLM-L6-v2' loaded successfully.
Model details: 384 dimensions


### VectorStore

In [8]:
class VectorStore:
    """ Manages a vector store using ChromaDB for storing and retrieving document embeddings. """
    
    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vectore_store"):
        """
        Initialize the VectorStore with a specified collection name and persistence directory.

        Args:
            collection_name : Name of the ChromaDB collection to use. Default is "pdf_documents".
            persist_directory : Directory to persist the ChromaDB data. Default is "../data/vectore_store".
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()
        
    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
        
vector_store = VectorStore()
vector_store

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 35


In [9]:
### Convert the text chunks into embeddings and add them to the vector store
texts= [chunk.page_content for chunk in chunks]
vector_store.add_documents(chunks, embedding_manager.generate_embeddings(texts))


Generating embeddings for 35 texts...
Embeddings generated successfully. Shape: (35, 384)
Adding 35 documents to vector store...
Successfully added 35 documents to vector store
Total documents in collection: 70


### Retriever Pipeline from vectorStore

In [10]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(vector_store, embedding_manager)

In [11]:
rag_retriever.retrieve("how should i prepare dsa for interviews ?")

Retrieving documents for query: 'how should i prepare dsa for interviews ?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Embeddings generated successfully. Shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c1a53d38_11',
  'content': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy',
  'metadata': {'page': 11,
   'source_file': 'BASIC_DSA_TRCIKS.pdf',
   'producer': 'Aspose.PDF for .NET 21.5.0',
   'content_length': 167,
   'creator': 'Aspose Ltd.',
   'doc_index': 11,
   'creationdate': '2023-09-22T17:37:59+05:30',
   'moddate': '2023-09-22T18:03:42+05:30',
   'page_label': '12',
   'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf',
   'file_type': 'pdf',
   'total_pages': 12},
  'similarity_score': 0.17406094074249268,
  'distance': 0.8259390592575073,
  'rank': 1},
 {'id': 'doc_c3e8010d_11',
  'content': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy',
  'metadata': {'creationdate': '2023-09-22T17:37:59+05:30',
   'total_p

In [12]:
rag_retriever.retrieve("dsa preparation tips")

Retrieving documents for query: 'dsa preparation tips'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...
Embeddings generated successfully. Shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c1a53d38_11',
  'content': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy',
  'metadata': {'file_type': 'pdf',
   'creator': 'Aspose Ltd.',
   'creationdate': '2023-09-22T17:37:59+05:30',
   'moddate': '2023-09-22T18:03:42+05:30',
   'content_length': 167,
   'producer': 'Aspose.PDF for .NET 21.5.0',
   'source_file': 'BASIC_DSA_TRCIKS.pdf',
   'total_pages': 12,
   'doc_index': 11,
   'page_label': '12',
   'page': 11,
   'source': '../data/pdf_files/BASIC_DSA_TRCIKS.pdf'},
  'similarity_score': 0.11631035804748535,
  'distance': 0.8836896419525146,
  'rank': 1},
 {'id': 'doc_c3e8010d_11',
  'content': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy',
  'metadata': {'creator': 'Aspose Ltd.',
   'source': '../data/pdf_file

### Integration VectorDb Context Pipeline With LLM Output

In [13]:
### Simple RAG pipeline that takes a query, retrieves relevant documents, and returns the top results
import os
from dotenv import load_dotenv

load_dotenv()

from langchain_openai import ChatOpenAI

custom_llm = ChatOpenAI(
    model="minimax-local",
    api_key=os.getenv("CHAT_API_KEY"),
    base_url=os.getenv("CHAT_API_BASE_URL"),
    temperature=0,
    verbose=True
)

In [14]:
def rag_simple(query, retriever, top_k=3):
    # retrieve the context from the vector store
    retrieved_docs = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in retrieved_docs]) if retrieved_docs else ""
    
    if not context:
        print("No relevant documents found for the query.")
        return "No relevant documents found."
    
    # create a prompt for the LLM
    prompt = f"""
            You are a helpful assistant. Use the following context to answer the question concisely.
            Context: {context}
            Question: {query}   
            Answer:
            """ 

    response = custom_llm.invoke([prompt.format(context=context, query=query)] )
    return response.content

In [15]:
answer = rag_simple("how should i prepare dsa for interviews ?", rag_retriever, top_k=3)

Retrieving documents for query: 'how should i prepare dsa for interviews ?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...
Embeddings generated successfully. Shape: (1, 384)
Retrieved 2 documents (after filtering)


In [16]:
answer

'To prepare for DSA interviews, focus on:\n\n1. **Master core topics** – Arrays, Strings, Linked Lists, Stacks, Queues, Trees, Graphs, Heaps, Hashing, Recursion, Dynamic Programming, and Sorting/Searching.\n2. **Practice daily** – Solve 2–3 problems on platforms like LeetCode, GeeksforGeeks, or HackerRank, starting with easy, then medium, then hard.\n3. **Understand patterns** – Learn common problem-solving patterns (sliding window, two pointers, BFS/DFS, backtracking, etc.).\n4. **Time & space complexity** – Always analyze and optimize your solutions.\n5. **Mock interviews** – Practice with peers or use platforms like Pramp to simulate real interviews.\n6. **Revise regularly** – Maintain notes of solved problems and revisit them weekly.\n\nFor a structured, guided approach, consider joining a comprehensive program like **Logicmojo’s 7-month DSA + System Design (HLD+LLD) live classes**, which covers these topics in depth and helps you crack tech interviews effectively.'

### Enhance the rag pipeline

In [18]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = custom_llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("DSA Interview Preparation", rag_retriever, custom_llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'DSA Interview Preparation'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Embeddings generated successfully. Shape: (1, 384)
Retrieved 2 documents (after filtering)
Answer: Based on the context, the answer is to **join Logicmojo Live Classes**, a 7-month program covering DSA + System Design (HLD+LLD) to crack upcoming tech interviews.
Sources: [{'source': 'BASIC_DSA_TRCIKS.pdf', 'page': 11, 'score': 0.23297017812728882, 'preview': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy...'}, {'source': 'BASIC_DSA_TRCIKS.pdf', 'page': 11, 'score': 0.23297017812728882, 'preview': '➡ Learn these Tips & techniques \nJoin Logicmojo Live Classes \n7 month Program for \nDSA + System Design(HLD+LLD)\nCrack your upcoming Tech \nInterviews \nLogicmojo Academy...'}]
Confidence: 0.23297017812728882
Context Preview: ➡ Learn thes

In [22]:
result = rag_advanced("DSA TEchniques", rag_retriever, custom_llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'DSA TEchniques'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...
Embeddings generated successfully. Shape: (1, 384)
Retrieved 0 documents (after filtering)
Answer: No relevant context found.
Sources: []
Confidence: 0.0
Context Preview: 
